<a href="https://colab.research.google.com/github/jeombagis/Stock-price-prediction/blob/main/Daily_Data_Extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import csv, time, requests
import os
from datetime import datetime, timedelta
from google.colab import drive

In [ ]:
# 1. 설정 정보
BASE = "https://mockapi.kiwoom.com"      # 실전투자 https://api.kiwoom.com (모의투자는 https://mockapi.kiwoom.com)
API_ID = "ka10081"                   # 주식일봉차트조회 TR 코드
#SYMBOL = "005930"                    # 삼성전자
#SYMBOL = "000660"                    # SK 하이닉스
SYMBOL = "039490"                    # 키움증권

In [ ]:
def get_token():
    # 실제 발급받은 토큰 문자열을 반환하도록 수정하세요
    return "2Ek8J5jwEUEsIF1OAmtrD3VAraHLqeyLyr2QCRr9XSn1X24UlQwOZSHg3q6kBVCBJ2WO5f4VKk29u9RjGHMXNw"

In [ ]:
def fetch_20_years_daily(symbol: str):
    url = f"{BASE}/api/dostk/chart"

    # [수정] 오늘이 주말일 수 있으므로 확실히 데이터가 있는 '최근 영업일'부터 시작
    start_dt = (datetime.now() - timedelta(days=2)).strftime('%Y%m%d')
    target_date = (datetime.now() - timedelta(days=365 * 20)).strftime('%Y%m%d')

    # [수정] 종목코드에서 KRX:를 제거한 순수 숫자 6자리만 사용 (서버 특성에 따라 다름)
    payload = {
        "stk_cd": symbol,
        "base_dt": start_dt,
        "upd_stkpc_tp": "1",
    }

    all_data = []
    next_key, cont_yn = "", "N"

    print(f"--- [20년치 수집 시작] 시작일: {start_dt}, 목표: {target_date} ---")

    while True:
        headers = {
            "Authorization": f"Bearer {get_token().strip()}",
            "Content-Type": "application/json;charset=UTF-8",
            "api-id": API_ID,
            "cont-yn": cont_yn,
            "next-key": next_key
        }

        try:
            r = requests.post(url, headers=headers, json=payload, timeout=20)
            r.raise_for_status()
            data = r.json()

            # 데이터 리스트 추출
            rows = data.get("stk_dt_pole_chart_qry", [])

            if not rows:
                print("⚠️ 데이터를 더 이상 가져올 수 없습니다.")
                break

            all_data.extend(rows)

            # 마지막 데이터 날짜 확인
            last_date = str(rows[-1].get('dt', '')).strip()
            print(f"▶ {len(rows)}건 추가 (누적: {len(all_data)}건) / 마지막 날짜: {last_date}")

            # 목표 날짜 도달 시 종료
            if last_date and last_date <= target_date:
                print("✅ 20년치 데이터 수집 완료!")
                break

            # [핵심] 응답 헤더에서 다음 페이지 정보 갱신
            cont_yn = r.headers.get("cont-yn", "N")
            next_key = r.headers.get("next-key", "")

            # 연속 데이터 신호가 없으면 종료
            if cont_yn == "N" or not next_key:
                print("--- 서버 신호: 다음 데이터 없음 ---")
                break

            time.sleep(1.2) # API 호출 제한 방지

        except Exception as e:
            print(f"❌ 오류: {e}")
            break

    return all_data

In [ ]:
if __name__ == "__main__":
    # 1. 구글 드라이브 마운트
    drive.mount('/content/drive')

    # 2. [수정됨] 사용자 지정 저장 경로 설정
    save_dir = '/content/drive/MyDrive/StockPricePrediction/'
    os.makedirs(save_dir, exist_ok=True) # 폴더가 없으면 에러 없이 자동 생성해 줍니다.

    print(f"📥 '{SYMBOL}' 종목 데이터 수집을 시작합니다...")
    data_rows = fetch_20_years_daily(SYMBOL)

    if data_rows:
        file_name = f"{SYMBOL}_20years_daily.csv"
        save_path = os.path.join(save_dir, file_name)

        # 3. CSV 파일로 저장
        with open(save_path, "w", newline="", encoding="utf-8-sig") as f:
            w = csv.writer(f)
            # 헤더 작성
            w.writerow(["date", "open", "high", "low", "close", "volume"])
            # 데이터 작성
            for r in data_rows:
                w.writerow([r.get("dt"), r.get("open_pric"), r.get("high_pric"),
                            r.get("low_pric"), r.get("cur_prc"), r.get("trde_qty")])

        print(f"\n🎉 [성공] 총 {len(data_rows)}행 저장 완료!")
        print(f"📁 최종 저장 위치: {save_path}")
    else:
        print("❌ 수집 실패: 토큰 또는 종목코드를 다시 확인해 주세요.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📥 '039490' 종목 데이터 수집을 시작합니다...
--- [20년치 수집 시작] 시작일: 20260309, 목표: 20060316 ---
▶ 600건 추가 (누적: 600건) / 마지막 날짜: 20230913
▶ 600건 추가 (누적: 1200건) / 마지막 날짜: 20210413
▶ 600건 추가 (누적: 1800건) / 마지막 날짜: 20181106
▶ 600건 추가 (누적: 2400건) / 마지막 날짜: 20160525
▶ 600건 추가 (누적: 3000건) / 마지막 날짜: 20131212
▶ 600건 추가 (누적: 3600건) / 마지막 날짜: 20110714
▶ 600건 추가 (누적: 4200건) / 마지막 날짜: 20090224
▶ 600건 추가 (누적: 4800건) / 마지막 날짜: 20060915
▶ 600건 추가 (누적: 5400건) / 마지막 날짜: 20040423
✅ 20년치 데이터 수집 완료!

🎉 [성공] 총 5400행 저장 완료!
📁 최종 저장 위치: /content/drive/MyDrive/StockPricePrediction/039490_20years_daily.csv


In [ ]:
# ==========================================
# 로컬 실행용 셀
# ==========================================
if __name__ == "__main__":
    current_dir = os.getcwd()
    data_rows = fetch_20_years_daily(SYMBOL)

    if data_rows:
        file_name = f"{SYMBOL}_20years_daily.csv"
        save_path = os.path.join(current_dir, file_name)

        with open(save_path, "w", newline="", encoding="utf-8-sig") as f:
            w = csv.writer(f)
            w.writerow(["date", "open", "high", "low", "close", "volume"])
            for r in data_rows:
                w.writerow([r.get("dt"), r.get("open_pric"), r.get("high_pric"),
                            r.get("low_pric"), r.get("cur_prc"), r.get("trde_qty")])

        print(f"\n[성공] 총 {len(data_rows)}행 저장 완료 -> {file_name}")
    else:
        print("❌ 수집 실패: 토큰 또는 종목코드를 확인하세요.")